In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import MNIST


In [20]:
#DATASETS & DATALOADER
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

trainset = MNIST(root = "./data", train = True, download = True, transform = transform)
testset = MNIST(root = "./data", train = False, download = True, transform = transform)

In [21]:
trainloader = DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = DataLoader(testset, batch_size = 64, shuffle = False)

BULIDING CNN

In [30]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(3*3*128, 256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)
        x = self.fc_layer(x)

        return x
            

In [31]:
model = CNN()

In [32]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [35]:
epochs = 10

for epoch in range(epochs):
    epoch_train_loss = 0.0

    for images,label in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criterion(output,label) #loss fnx
        loss.backward() #back propagation
        optimizer.step() #update params
        epoch_train_loss +=loss.item()

    print(f"epoch{epoch+1}/{epochs} ==>  loss = {epoch_train_loss/len(trainloader)}")

epoch1/10 ==>  loss = 0.04822836451265695
epoch2/10 ==>  loss = 0.032803934670104555
epoch3/10 ==>  loss = 0.02604446341017093
epoch4/10 ==>  loss = 0.02008404024517014
epoch5/10 ==>  loss = 0.015897336475674494
epoch6/10 ==>  loss = 0.013531898911295935
epoch7/10 ==>  loss = 0.012031408995997419
epoch8/10 ==>  loss = 0.01009836371792317
epoch9/10 ==>  loss = 0.010188433078562013
epoch10/10 ==>  loss = 0.007423450928315939


In [40]:
correct_label = 0
total = 0

model.eval()
with torch.no_grad():
    for images,label in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)
        correct_label += (predicted == label).sum().item()
        total += label.size(0)

print("Accuracy: ",correct_label/total*100)

Accuracy:  99.22999999999999
